In [ ]:

storage_account_name = "REPLACE_WITH_ACTUAL_KEY"
storage_account_key = "REPLACE_WITH_ACTUAL_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
    storage_account_key
)

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as f


In [0]:
# Define paths and load all bronze tables into DataFrames dynamically
bronze_path = "abfss://olistdb@iliststorageaccount.dfs.core.windows.net/bronze/"
silver_path = "abfss://olistdb@iliststorageaccount.dfs.core.windows.net/silver/"


tables_ds = [{"bronze_customers":bronze_path+"olist_customers_dataset.csv",
             "bronze_geolocation":bronze_path+"olist_geolocation_dataset.csv",
             "bronze_order_items":bronze_path+"olist_order_items_dataset.csv",
             "bronze_order_payments":bronze_path+"olist_order_payments",
             "bronze_order_reviews":bronze_path+"olist_order_reviews_dataset.csv",
             "bronze_orders":bronze_path+"olist_orders_dataset.csv",
             "bronze_products":bronze_path+"olist_products_dataset.csv",
             "bronze_sellers":bronze_path+"olist_sellers_dataset.csv",
             "bronze_product_category_name_translation":bronze_path+"product_category_name_translation.csv"}
             ]

def write_df(table_path):
    df_name=spark.read.csv(table_path,header=True,inferSchema=True,multiLine=True)
    return df_name

for table in tables_ds:
    for table_name , path in table.items():
        globals()[table_name] = write_df(path)

In [0]:
def null_check(df_name):
    print(df_name)
    new_df = df_name.select([count(when(col(c).isNull(),1)).alias(c) for c in df_name.columns])
    new_df.show(n=1 , vertical=True)

for table in tables_ds:
    for table_name , path in table.items():
        df_name = globals().get(table_name)
        null_check(df_name)


In [0]:
# Aggregate geolocation data by zip code to calculate average latitude and longitude

silver_geolocation = (bronze_geolocation
                      .filter(col('geolocation_zip_code_prefix').isNotNull())
                      .groupBy('geolocation_zip_code_prefix').agg(avg('geolocation_lat').alias('geolocation_lat'),
                            avg('geolocation_lng').alias('geolocation_lng'))        
                        )


In [0]:
# Clean customer data: remove duplicates, trim strings, and add geolocation coordinates
silver_customers =  (bronze_customers
                     
                     ## filtering (null handling)
                     .filter(col('customer_unique_id').isNotNull())
                
                     ## deduplicatinon
                     .dropDuplicates(["customer_id"])
 

                     ## standerization (trim)
                     .select(*[
                             trim(c).alias(c) if bronze_customers.schema[c].dataType.typeName()=='string'
                             else col(c)
                             for c in bronze_customers.columns
                     ])
                     
                     ## metadata  (join ,  add columns)
                     .join(silver_geolocation.select('geolocation_lat' ,'geolocation_zip_code_prefix', "geolocation_lng") , bronze_customers.customer_zip_code_prefix==silver_geolocation.geolocation_zip_code_prefix , how='left' )
                     .drop('geolocation_zip_code_prefix')
                     .withColumn('created_at', current_timestamp())
                     )

In [0]:
# Clean sellers data: remove duplicates and join with geolocation coordinates
silver_sellers  =  (
                        bronze_sellers
                        .filter(col('seller_id').isNotNull())

                        .dropDuplicates(["seller_id"])

                        .select(*[
                            trim(c).alias(c) if bronze_sellers.schema[c].dataType.typeName() == 'string'
                            else col(c)
                            for c in bronze_sellers.columns
                            ])
                                

                        .join(silver_geolocation.select('geolocation_lat' ,'geolocation_zip_code_prefix', "geolocation_lng") , bronze_sellers.seller_zip_code_prefix==silver_geolocation.geolocation_zip_code_prefix , how='left').drop('geolocation_zip_code_prefix')
                        .withColumn('created_at', current_timestamp())             
                     )




In [0]:
silver_product_category_name_translation = (

                                            bronze_product_category_name_translation
                                            .filter(col('product_category_name').isNotNull())
                                            .filter(col('product_category_name_english').isNotNull())

                                            .dropDuplicates(["product_category_name"])

)



In [0]:
silver_products = ( bronze_products
                  .filter(col('product_id').isNotNull())
                  .filter(col('product_category_name').isNotNull())

                  .dropDuplicates(["product_id"])

                  .select(*[
                            trim(c).alias(c) if bronze_products.schema[c].dataType.typeName() == 'string'
                            else col(c)
                            for c in bronze_products.columns
                            ])
                  .join(silver_product_category_name_translation , on="product_category_name",how="left").drop("product_category_name")

                  .withColumn('created_at', current_timestamp())              
                  )




In [0]:
# Transform orders data: calculate delivery time metrics, cast date columns, and flag issues

silver_orders = (bronze_orders
                 .filter(col('order_id').isNotNull())
                 
                 .dropDuplicates()

                 .select(*[trim(c).alias(c) if bronze_orders.schema[c].dataType.typeName() == 'string'
                           else col(c)
                           for c in bronze_orders.columns
                           ])

                 .withColumn("approved_performance_hour", f.round(((f.col("order_approved_at").cast("long") - f.col("order_purchase_timestamp").cast("long")) / 3600), 2))
                 .withColumn('total_process_days', datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp')))
                 .withColumn('delivery_performance_days', datediff(col('order_estimated_delivery_date'), col('order_delivered_customer_date')))

                 .withColumn('order_purchase_date', col('order_purchase_timestamp').cast('Date'))
                 .withColumn('order_approved_date', col('order_approved_at').cast('Date'))
                 .withColumn('order_delivered_carrier_date_d', col('order_delivered_carrier_date').cast('Date'))
                 .withColumn('order_delivered_customer_date_d', col('order_delivered_customer_date').cast('Date'))

                 .withColumn('order_purchase_time', date_format(col('order_purchase_timestamp'), "HH:mm:ss"))
                 .withColumn('order_approved_time', date_format(col('order_approved_at'), "HH:mm:ss"))
                 .withColumn('order_delivered_carrier_time', date_format(col('order_delivered_carrier_date'), "HH:mm:ss"))
                 .withColumn('order_delivered_customer_time', date_format(col('order_delivered_customer_date'), "HH:mm:ss"))

                 .withColumn('red_flag', 
                             when(
                                 (
                                     (col('order_status') == 'delivered') & 
                                     (
                                         col('order_delivered_customer_date').isNull() |
                                         col("order_approved_at").isNull() |
                                         col('order_delivered_carrier_date').isNull() |
                                         col('order_purchase_timestamp').isNull() |
                                         col('order_estimated_delivery_date').isNull()
                                     )
                                 ) | 
                                 (
                                     (col('order_status') == 'canceled') & 
                                     (col('order_delivered_customer_date').isNotNull())
                                 ) | 
                                 (col('order_delivered_customer_date') < col('order_purchase_timestamp')), 
                                 lit(1)
                             ).otherwise(lit(0))
                 )

                 .withColumn('created_at', current_timestamp())              
                 )


In [0]:

silver_quarantine = bronze_orders.filter(
    col('order_id').isNull() | 
    col('customer_id').isNull() | 
    col('order_status').isNull()
)

In [0]:
silver_order_payments = (bronze_order_payments
                        .filter(col('order-_id').isNotNull())
                        .withColumn('created_at', current_timestamp())
                        .withColumn('payment_sequential' , 
                                    when(
                                        col('payment_sequential')==0
                                        , lit(1)
                                        ).otherwise(col('payment_sequential'))
                                    )
                        .withColumn('payment_installments' , 
                                    when(
                                        col('payment_installments')==0
                                        , lit(1)
                                        ).otherwise(col('payment_installments'))
                                    )

                        
                        )

In [0]:
# Process reviews: cast data types and use Window functions to keep only the latest review per order
window_spec_reviews = Window.partitionBy('order_id').orderBy(col('review_answer_timestamp').desc())
silver_order_reviews = (bronze_order_reviews
                        .withColumn('review_score',expr("try_cast(review_score as int)"))
                        .withColumn('review_answer_timestamp',expr("try_cast(review_answer_timestamp as date)"))
                        .withColumn('review_creation_date',expr("try_cast(review_creation_date as date)"))

                        
                        .filter(col('order_id').isNotNull())
                        .filter(col('review_score').between(1,5))

                        .withColumn('rank',row_number().over(window_spec_reviews))
                        .filter(col('rank')==1).drop('rank')

                        .withColumn('created_at', current_timestamp())
                        

                        )




In [0]:
# Aggregate order items: calculate total quantity, combined generic prices, and total freight value per order

silver_order_items = (
    bronze_order_items

    .dropDuplicates()

    .select(*[
        trim(c).alias(c) if bronze_order_items.schema[c].dataType.typeName() == "string"
        else c
        for c in bronze_order_items.columns
    ])


    .withColumn('order_item_id', expr("try_cast(order_item_id as int)"))
    .withColumn('shipping_limit_date', expr("try_cast(shipping_limit_date as timestamp)"))
    .withColumn('price', expr("try_cast(price as DECIMAL(10,2))"))
    .withColumn('freight_value',expr("try_cast(freight_value as DECIMAL(10,2))"))


    .groupby('order_id','product_id')
    .agg(
            f.count('*').alias('Total_QTY'),
            f.first('price').alias('Unit_price'),
            f.first('freight_value').alias('Unit_freight'),
            f.first('seller_id').alias('seller_id'),
            f.first('shipping_limit_date').alias('shipping_limit_date')
    )
    .withColumn('Total_product_price',col('Unit_price')*col('Total_QTY'))
    .withColumn('Total_freight',col('Unit_freight')*col('Total_QTY'))
    .withColumn('Total_order_value',col('Total_product_price')+col('Total_freight'))

    .withColumn('created_at', current_timestamp())

)



In [0]:
# Overwrite existing tables and save the cleaned DataFrames as Delta tables in the Silver layer
df_list = [
    (silver_order_reviews, "silver_order_reviews"),
    (silver_order_payments, "silver_order_payments"),
    (silver_order_items, "silver_order_items"),
    (silver_quarantine, "silver_quarantine_orders"),
    (silver_orders, "silver_orders"),
    (silver_products, "silver_products"),
    (silver_product_category_name_translation, "silver_product_category_translation"),
    (silver_sellers, "silver_sellers"),
    (silver_customers, "silver_customers"),
    (silver_geolocation, "silver_geolocation")
]

for df, table_name in df_list:
    spark.sql(f"DROP TABLE IF EXISTS default.{table_name}")
    df.write.format("delta").option("path",silver_path+table_name).mode("overwrite").saveAsTable(table_name)
    print(f"Table '{table_name}' has been created as Table.")